# RO-051 Rapidly-exploring random trees: the numbers

Every number quoted in the Note, computed here. The planners live in `images/rrt.py`, on the disk robot's C-space of the room from `images/cspace.py`.

In [1]:
import sys
sys.path.insert(0, 'images')
import numpy as np
import rrt, cspace as c
np.set_printoptions(precision=4, suppress=True)
print('start', rrt.START, 'goal', rrt.GOAL, 'step eta', rrt.ETA, 'edge resolution', rrt.EPS, 'goal bias', rrt.GOAL_BIAS)
L_star, P_star = rrt.shortest_route()
print('shortest route from under the table:', round(L_star, 3), 'm through', len(P_star), 'points')
print('grid points 0.05 m apart in the room:', 141 * 81, '| with a heading every 1 degree:', 141 * 81 * 360)
print('six joints, 360 values each:', f'{360.0**6:.2e}', '| LaValle: 50 points per axis, 10 dimensions:', f'{50.0**10:.1e}')

start [4.  0.5] goal [3.6 4. ] step eta 0.5 edge resolution 0.05 goal bias 0.05


shortest route from under the table: 4.547 m through 20 points
grid points 0.05 m apart in the room: 11421 | with a heading every 1 degree: 4111560
six joints, 360 values each: 2.18e+15 | LaValle: 50 points per axis, 10 dimensions: 9.8e+16


## RRT, step by step (seed 22)

In [2]:
rrt.Counter.checks = 0
r = rrt.rrt(np.random.default_rng(22), log=True)
P = rrt.path_from(r['nodes'], r['parent'], r['goal'])
print('samples', r['samples'], 'nodes', len(r['nodes']), 'rejected', sum(not e['ok'] for e in r['events']),
      'collision checks', rrt.Counter.checks, 'route', round(rrt.length(P), 3), 'm with', len(P) - 1, 'edges')
for e in r['events'][:4]:
    q_near = r['nodes'][e['near']] if e['near'] < len(r['nodes']) else None
    d = np.hypot(*(e['q_rand'] - q_near))
    print(e['i'], 'sample', e['q_rand'].round(3), '| nearest node', e['near'], q_near.round(3), 'at', round(d, 3),
          'm | new', e['q_new'].round(3), '| free' if e['ok'] else '| rejected')
e = r['events'][0]
u = (e['q_rand'] - rrt.START) / np.hypot(*(e['q_rand'] - rrt.START))
print('direction', (e['q_rand'] - rrt.START).round(3), 'unit', u.round(4), 'step', (0.5 * u).round(3))
print('points checked on one 0.5 m edge:', int(np.ceil(0.5 / rrt.EPS)) + 1)
print('goal samples at:', [x['i'] for x in r['events'] if x['goal_sample']])

samples 67 nodes 41 rejected 28 collision checks 465 route 6.077 m with 13 edges
1 sample [1.395 0.354] | nearest node 0 [4.  0.5] at 2.609 m | new [3.501 0.472] | free
2 sample [3.215 3.951] | nearest node 1 [3.501 0.472] at 3.49 m | new [3.46 0.97] | rejected
3 sample [5.859 0.206] | nearest node 0 [4.  0.5] at 1.882 m | new [4.494 0.422] | free
4 sample [4.252 0.201] | nearest node 2 [4.494 0.422] at 0.328 m | new [4.252 0.201] | rejected
direction [-2.605 -0.146] unit [-0.9984 -0.0559] step [-0.499 -0.028]
points checked on one 0.5 m edge: 11
goal samples at: [67]


In [3]:
S, L, S2, L2 = [], [], [], []
for seed in range(200):
    a = rrt.rrt(np.random.default_rng(seed)); S.append(a['samples'])
    L.append(rrt.length(rrt.path_from(a['nodes'], a['parent'], a['goal'])))
    b = rrt.rrt_connect(np.random.default_rng(seed)); S2.append(b['samples']); L2.append(rrt.length(rrt.connect_path(b)))
print('RRT over 200 seeds: samples median', np.median(S), 'route median', round(np.median(L), 2))
print('RRT-Connect over 200 seeds: rounds median', np.median(S2), 'route median', round(np.median(L2), 2))

RRT over 200 seeds: samples median 65.0 route median 6.07
RRT-Connect over 200 seeds: rounds median 27.0 route median 5.61


## Why random samples explore: RRT against a random-walk tree

In [4]:
cr = [rrt.coverage(rrt.rrt(np.random.default_rng(s), n_max=100000, stop_at_goal=False, bias=0.0, n_nodes=100)['nodes']) for s in range(20)]
cw = [rrt.coverage(rrt.walk_tree(np.random.default_rng(s), 100)['nodes']) for s in range(20)]
print('share of free floor within 0.5 m of a node after 100 nodes: RRT', round(np.mean(cr), 3), 'walk', round(np.mean(cw), 3))
print('seed 0: RRT', round(cr[0], 3), 'walk', round(cw[0], 3))
print('free area of the room for the robot centre:', round(rrt.free_area(), 2), 'm2')

share of free floor within 0.5 m of a node after 100 nodes: RRT 0.946 walk 0.274
seed 0: RRT 0.92 walk 0.171
free area of the room for the robot centre: 15.02 m2


## RRT keeps its first route; RRT* shortens it

In [5]:
REC = (100, 150, 200, 300, 400, 500, 750, 1000, 1500, 2000, 3000, 4000)
plain = np.median([[o[k] for k in REC] for o in (rrt.rrt_best(np.random.default_rng(s), REC) for s in range(20))], axis=0)
star = np.median([[x['best'][k] for k in REC] for x in (rrt.rrt_star(np.random.default_rng(s), n_max=4000, record=REC) for s in range(20))], axis=0)
for k, a, b in zip(REC, plain, star):
    print(k, 'RRT best so far', round(a, 3), '| RRT*', round(b, 3))

100 RRT best so far 6.477 | RRT* 6.108
150 RRT best so far 6.073 | RRT* 5.734
200 RRT best so far 6.073 | RRT* 5.481
300 RRT best so far 6.073 | RRT* 5.229
400 RRT best so far 6.073 | RRT* 5.141
500 RRT best so far 6.073 | RRT* 5.013
750 RRT best so far 6.073 | RRT* 4.827
1000 RRT best so far 6.073 | RRT* 4.783
1500 RRT best so far 6.073 | RRT* 4.731
2000 RRT best so far 6.073 | RRT* 4.706
3000 RRT best so far 6.073 | RRT* 4.677
4000 RRT best so far 6.073 | RRT* 4.656


In [6]:
A = rrt.free_area()
gmin = np.sqrt(3) * np.sqrt(A / np.pi)
print('gamma lower bound for d = 2:', round(gmin, 3), '| used', rrt.GAMMA)
for n in (100, 380, 1000, 2000):
    print('radius with', n, 'nodes:', round(rrt.radius(n), 3))
full = rrt.rrt_star(np.random.default_rng(3), n_max=2000, record=tuple(range(1, 2001)))
first = min(k for k, v in full['best'].items() if np.isfinite(v))
print('seed 3: first route at sample', first, round(full['best'][first], 3), '| 200:', round(full['best'][200], 3),
      '| 500:', round(full['best'][500], 3), '| 1000:', round(full['best'][1000], 3), '| 2000:', round(full['best'][2000], 3),
      '| nodes', len(full['nodes']))

gamma lower bound for d = 2: 3.787 | used 4.0
radius with 100 nodes: 0.5
radius with 380 nodes: 0.5
radius with 1000 nodes: 0.332
radius with 2000 nodes: 0.247


seed 3: first route at sample 63 6.608 | 200: 6.318 | 500: 5.145 | 1000: 4.87 | 2000: 4.789 | nodes 1087


In [7]:
r0 = rrt.rrt_star(np.random.default_rng(3), n_max=118)
r1 = rrt.rrt_star(np.random.default_rng(3), n_max=119, log=True)
e = r1['events'][-1]
N, Cst = r0['nodes'], r0['cost']
print('sample 119:', e['q_rand'].round(3), 'nearest', e['near'], 'new node', e['q_new'].round(3), 'radius', e['r'])
for j in e['near_set']:
    d = np.hypot(*(N[j] - e['q_new']))
    print('  near node', j, N[j].round(3), 'cost', round(Cst[j], 4), 'edge', round(d, 4), 'through it', round(Cst[j] + d, 4))
print('parent chosen', e['parent'], 'rewired (node, old parent)', e['rewired'])
print('route', round(Cst[r0['goal']], 4), '->', round(r1['cost'][r1['goal']], 4))

sample 119: [2.41  3.212] nearest 16 new node [2.41  3.212] radius 0.5
  near node 12 [2.171 2.913] cost 4.2196 edge 0.3827 through it 4.6023
  near node 16 [2.638 3.09 ] cost 4.7108 edge 0.259 through it 4.9698
  near node 21 [2.661 3.589] cost 5.2108 edge 0.4539 through it 5.6646
parent chosen 12 rewired (node, old parent) [(21, 16)]
route 6.5994 -> 6.4448


In [8]:
import time
for name, f in (('RRT', lambda s: rrt.rrt(np.random.default_rng(s), n_max=2000, stop_at_goal=False)),
                ('RRT*', lambda s: rrt.rrt_star(np.random.default_rng(s), n_max=2000))):
    ch, tt = [], []
    for s in range(5):
        rrt.Counter.checks = 0; t = time.time(); f(s); tt.append(time.time() - t); ch.append(rrt.Counter.checks)
    print(name, '2,000 samples: collision checks (mean of 5)', round(np.mean(ch)), '| per sample', round(np.mean(ch) / 2000, 1))

RRT 2,000 samples: collision checks (mean of 5) 6189 | per sample 3.1


RRT* 2,000 samples: collision checks (mean of 5) 23834 | per sample 11.9


## RRT-Connect (seed 38)

In [9]:
cc = rrt.rrt_connect(np.random.default_rng(38), log=True)
print('rounds', cc['samples'], 'start tree', len(cc['Ts'][0]), 'goal tree', len(cc['Tg'][0]),
      'route', round(rrt.length(rrt.connect_path(cc)), 3))
print('rounds that added a node:', [(x['i'], 'start' if x['a_is_start'] else 'goal', x['added_b']) for x in cc['events'] if x['added_a']])

rounds 27 start tree 5 goal tree 13 route 5.403
rounds that added a node: [(2, 'goal', 0), (3, 'start', 1), (15, 'start', 0), (18, 'goal', 0), (20, 'goal', 0), (22, 'goal', 0), (24, 'goal', 0), (25, 'start', 0), (26, 'goal', 0), (27, 'start', 5)]
